# 11_Doo — SVC 하이퍼파라미터 튜닝

## 실험 원칙
Train 내부 5-fold 교차검증으로 튜닝하고 Validation에서 최종 비교한다. Test 데이터는 사용하지 않는다. `random_state=42`를 유지한다.

## 1. 데이터와 베이스라인
SVC는 거리 기반 모델이므로 팀 전처리에서 스케일링된 데이터를 사용한다. RBF 커널 SVC에 확률 보정을 적용해 이탈 확률을 계산한다.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score

root = Path.cwd()
while not (root / 'data' / 'preprocessed').exists():
    if root.parent == root:
        raise FileNotFoundError('data/preprocessed 폴더를 찾을 수 없음')
    root = root.parent
data_dir = root / 'data' / 'preprocessed'
X_train = pd.read_csv(data_dir / 'X_train.csv')
y_train = pd.read_csv(data_dir / 'y_train.csv')['churn']
X_val = pd.read_csv(data_dir / 'X_val.csv')
y_val = pd.read_csv(data_dir / 'y_val.csv')['churn']
print('X_train:', X_train.shape, '| X_val:', X_val.shape)

def make_svc(C=1.0, gamma='scale', class_weight='balanced'):
    base = SVC(C=C, kernel='rbf', gamma=gamma,
               class_weight=class_weight, random_state=42)
    return CalibratedClassifierCV(base, method='sigmoid', cv=5, ensemble=False)

def evaluate(model, X, y):
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]
    return {'accuracy': accuracy_score(y, predictions), 'recall': recall_score(y, predictions), 'precision': precision_score(y, predictions, zero_division=0), 'f1': f1_score(y, predictions, zero_division=0), 'roc_auc': roc_auc_score(y, probabilities)}


X_train: (2592, 10) | X_val: (864, 10)


## 2. 하이퍼파라미터 튜닝
결정경계 벌점 `C`, 커널 영향 범위 `gamma`, 클래스 가중치를 Train 내부 5-fold Stratified CV에서 ROC-AUC 기준으로 탐색한다.

In [4]:
baseline = make_svc()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_distributions = {
    'estimator__C': np.logspace(-2, 2, 20),
    'estimator__gamma': ['scale', 'auto'] + list(np.logspace(-3, 0, 10)),
    'estimator__class_weight': ['balanced', None],
}
search = RandomizedSearchCV(
    make_svc(), param_distributions, n_iter=20, scoring='roc_auc',
    cv=cv, random_state=42, n_jobs=1, return_train_score=True,
)
search.fit(X_train, y_train)
tuned = search.best_estimator_
print('Best CV ROC-AUC:', round(search.best_score_, 4))
print('Best parameters:', search.best_params_)
baseline.fit(X_train, y_train)
rows = []
for name, model in [('Baseline', baseline), ('Tuned', tuned)]:
    train_metrics = evaluate(model, X_train, y_train)
    val_metrics = evaluate(model, X_val, y_val)
    rows.append({'model': name, 'train_accuracy': train_metrics['accuracy'], 'val_accuracy': val_metrics['accuracy'], 'recall': val_metrics['recall'], 'precision': val_metrics['precision'], 'f1': val_metrics['f1'], 'roc_auc': val_metrics['roc_auc'], 'train_val_gap': train_metrics['accuracy'] - val_metrics['accuracy']})
comparison = pd.DataFrame(rows).set_index('model')
display(comparison.round(3))


Best CV ROC-AUC: 0.7677
Best parameters: {'estimator__gamma': np.float64(0.021544346900318832), 'estimator__class_weight': 'balanced', 'estimator__C': np.float64(37.92690190732246)}


,train_accuracy,val_accuracy,recall,precision,f1,roc_auc,train_val_gap
model,,,,,,,
Baseline,0.715,0.708,0.719,0.699,0.709,0.763,0.007
Tuned,0.715,0.707,0.717,0.699,0.708,0.764,0.008


## 3. 임계값 비교
기본 임계값 0.50 외의 임계값을 비교한다. Recall 0.80 이상 후보 중 F1-score가 가장 높은 값을 선택한다.

In [5]:
tuned_proba = tuned.predict_proba(X_val)[:, 1]
threshold_rows = []
for threshold in np.arange(0.25, 0.71, 0.05):
    predictions = (tuned_proba >= threshold).astype(int)
    threshold_rows.append({'threshold': threshold, 'recall': recall_score(y_val, predictions), 'precision': precision_score(y_val, predictions, zero_division=0), 'f1': f1_score(y_val, predictions, zero_division=0), 'predicted_churn_count': int(predictions.sum())})
threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df.round(3))
candidates = threshold_df[threshold_df['recall'] >= 0.80]
selected = candidates.loc[candidates['f1'].idxmax()] if not candidates.empty else threshold_df.loc[threshold_df['f1'].idxmax()]
print('선택 임계값')
display(selected.to_frame('selected_value'))


,threshold,recall,precision,f1,predicted_churn_count
0,0.25,0.902,0.599,0.720,643
1,0.30,0.850,0.625,0.720,581
2,0.35,0.815,0.653,0.725,533
3,0.40,0.780,0.669,0.720,498
4,0.45,0.749,0.684,0.715,468
5,0.50,0.717,0.699,0.708,438
6,0.55,0.667,0.709,0.688,402
7,0.60,0.628,0.726,0.673,369
8,0.65,0.567,0.740,0.642,327
9,0.70,0.492,0.761,0.597,276


선택 임계값


,selected_value
threshold,0.350000
recall,0.814988
precision,0.652908
f1,0.725000
predicted_churn_count,533.000000


## 4. 결과 해석과 다음 단계
튜닝 전후 Validation 성능과 Train-Validation gap을 확인한다. 최종 모델 합의 후 Train+Validation 재학습과 Test 1회 평가를 진행한다.